## Investigate Metadata

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import Counter

# Adjust path if your kaggle download lands elsewhere
CSV_PATH = './Data_Entry_2017.csv'

df = pd.read_csv(CSV_PATH)

print("=== SHAPE ===")
print(df.shape)

print("\n=== COLUMNS ===")
print(df.columns.tolist())

print("\n=== DTYPES ===")
print(df.dtypes)

print("\n=== FIRST 3 ROWS ===")
print(df.head(3).to_string())

print("\n=== NULL COUNTS ===")
print(df.isnull().sum())

=== SHAPE ===
(112120, 12)

=== COLUMNS ===
['Image Index', 'Finding Labels', 'Follow-up #', 'Patient ID', 'Patient Age', 'Patient Gender', 'View Position', 'OriginalImage[Width', 'Height]', 'OriginalImagePixelSpacing[x', 'y]', 'Unnamed: 11']

=== DTYPES ===
Image Index                     object
Finding Labels                  object
Follow-up #                      int64
Patient ID                       int64
Patient Age                      int64
Patient Gender                  object
View Position                   object
OriginalImage[Width              int64
Height]                          int64
OriginalImagePixelSpacing[x    float64
y]                             float64
Unnamed: 11                    float64
dtype: object

=== FIRST 3 ROWS ===
        Image Index          Finding Labels  Follow-up #  Patient ID  Patient Age Patient Gender View Position  OriginalImage[Width  Height]  OriginalImagePixelSpacing[x     y]  Unnamed: 11
0  00000001_000.png            Cardiomegaly    

In [2]:
print("=== TOTAL IMAGES ===")
print(f"  {len(df):,}")

print("\n=== UNIQUE PATIENTS ===")
print(f"  {df['Patient ID'].nunique():,}")

print("\n=== IMAGES PER PATIENT (distribution) ===")
imgs_per_patient = df.groupby('Patient ID').size()
print(imgs_per_patient.describe().round(2))
print(f"  Patients with 1 image : {(imgs_per_patient == 1).sum():,}")
print(f"  Patients with 2-5     : {((imgs_per_patient >= 2) & (imgs_per_patient <= 5)).sum():,}")
print(f"  Patients with >5      : {(imgs_per_patient > 5).sum():,}")

print("\n=== GENDER DISTRIBUTION ===")
print(df['Patient Gender'].value_counts())
print(df['Patient Gender'].value_counts(normalize=True).round(3))

print("\n=== AGE DISTRIBUTION ===")
print(df['Patient Age'].describe().round(2))
print(f"  Age == 0 : {(df['Patient Age'] == 0).sum()}")
print(f"  Age > 100: {(df['Patient Age'] > 100).sum()}")

print("\n=== VIEW POSITION ===")
print(df['View Position'].value_counts())

=== TOTAL IMAGES ===
  112,120

=== UNIQUE PATIENTS ===
  30,805

=== IMAGES PER PATIENT (distribution) ===
count    30805.00
mean         3.64
std          7.27
min          1.00
25%          1.00
50%          1.00
75%          3.00
max        184.00
dtype: float64
  Patients with 1 image : 17,503
  Patients with 2-5     : 8,481
  Patients with >5      : 4,821

=== GENDER DISTRIBUTION ===
Patient Gender
M    63340
F    48780
Name: count, dtype: int64
Patient Gender
M    0.565
F    0.435
Name: proportion, dtype: float64

=== AGE DISTRIBUTION ===
count    112120.00
mean         46.90
std          16.84
min           1.00
25%          35.00
50%          49.00
75%          59.00
max         414.00
Name: Patient Age, dtype: float64
  Age == 0 : 0
  Age > 100: 16

=== VIEW POSITION ===
View Position
PA    67310
AP    44810
Name: count, dtype: int64


In [3]:
from itertools import chain

# Parse all labels
all_labels_raw = df['Finding Labels'].str.split('|')

# Flatten to get per-label counts
label_counts = Counter(chain.from_iterable(all_labels_raw))
label_counts_df = pd.DataFrame(
    label_counts.items(), columns=['Label', 'Count']
).sort_values('Count', ascending=False).reset_index(drop=True)

label_counts_df['Pct_of_images'] = (label_counts_df['Count'] / len(df) * 100).round(2)

print("=== LABEL FREQUENCY (all images, multi-label counted per label) ===")
print(label_counts_df.to_string(index=False))

print(f"\n=== TOTAL UNIQUE LABELS FOUND ===")
print(f"  {len(label_counts_df)}")

=== LABEL FREQUENCY (all images, multi-label counted per label) ===
             Label  Count  Pct_of_images
        No Finding  60361          53.84
      Infiltration  19894          17.74
          Effusion  13317          11.88
       Atelectasis  11559          10.31
            Nodule   6331           5.65
              Mass   5782           5.16
      Pneumothorax   5302           4.73
     Consolidation   4667           4.16
Pleural_Thickening   3385           3.02
      Cardiomegaly   2776           2.48
         Emphysema   2516           2.24
             Edema   2303           2.05
          Fibrosis   1686           1.50
         Pneumonia   1431           1.28
            Hernia    227           0.20

=== TOTAL UNIQUE LABELS FOUND ===
  15


In [4]:
# How many labels does each image have?
df['num_labels'] = df['Finding Labels'].apply(lambda x: len(x.split('|')))

print("=== LABELS PER IMAGE ===")
print(df['num_labels'].value_counts().sort_index().to_string())

print("\n=== AS PERCENTAGE ===")
print((df['num_labels'].value_counts(normalize=True).sort_index() * 100).round(2).to_string())

# Single-label images breakdown
single_label_df = df[df['num_labels'] == 1].copy()
print(f"\n=== SINGLE-LABEL IMAGES: {len(single_label_df):,} ===")
single_counts = single_label_df['Finding Labels'].value_counts()
single_counts_pct = (single_counts / len(single_label_df) * 100).round(2)
single_summary = pd.DataFrame({'Count': single_counts, 'Pct': single_counts_pct})
print(single_summary.to_string())

=== LABELS PER IMAGE ===
num_labels
1    91324
2    14306
3     4856
4     1247
5      301
6       67
7       16
8        1
9        2

=== AS PERCENTAGE ===
num_labels
1    81.45
2    12.76
3     4.33
4     1.11
5     0.27
6     0.06
7     0.01
8     0.00
9     0.00

=== SINGLE-LABEL IMAGES: 91,324 ===
                    Count    Pct
Finding Labels                  
No Finding          60361  66.10
Infiltration         9547  10.45
Atelectasis          4215   4.62
Effusion             3955   4.33
Nodule               2705   2.96
Pneumothorax         2194   2.40
Mass                 2139   2.34
Consolidation        1310   1.43
Pleural_Thickening   1126   1.23
Cardiomegaly         1093   1.20
Emphysema             892   0.98
Fibrosis              727   0.80
Edema                 628   0.69
Pneumonia             322   0.35
Hernia                110   0.12


In [5]:
single_label_df = df[df['num_labels'] == 1].copy()

print("=== GENDER × CLASS (single-label images) ===")
gender_class = single_label_df.groupby(['Finding Labels', 'Patient Gender']).size().unstack(fill_value=0)
gender_class['Total'] = gender_class.sum(axis=1)

# Add M/F ratio
if 'M' in gender_class.columns and 'F' in gender_class.columns:
    gender_class['M_pct'] = (gender_class['M'] / gender_class['Total'] * 100).round(1)
    gender_class['F_pct'] = (gender_class['F'] / gender_class['Total'] * 100).round(1)

print(gender_class.sort_values('Total', ascending=False).to_string())

=== GENDER × CLASS (single-label images) ===
Patient Gender          F      M  Total  M_pct  F_pct
Finding Labels                                       
No Finding          26439  33922  60361   56.2   43.8
Infiltration         4164   5383   9547   56.4   43.6
Atelectasis          1612   2603   4215   61.8   38.2
Effusion             1797   2158   3955   54.6   45.4
Nodule               1181   1524   2705   56.3   43.7
Pneumothorax         1193   1001   2194   45.6   54.4
Mass                  838   1301   2139   60.8   39.2
Consolidation         539    771   1310   58.9   41.1
Pleural_Thickening    466    660   1126   58.6   41.4
Cardiomegaly          585    508   1093   46.5   53.5
Emphysema             330    562    892   63.0   37.0
Fibrosis              340    387    727   53.2   46.8
Edema                 299    329    628   52.4   47.6
Pneumonia             128    194    322   60.2   39.8
Hernia                 70     40    110   36.4   63.6


In [6]:
# For Option B: single-label only
# Check if any patient appears in multiple classes — important for clean patient-level splits
single_label_df = df[df['num_labels'] == 1].copy()

patient_labels = single_label_df.groupby('Patient ID')['Finding Labels'].unique()
patient_num_classes = patient_labels.apply(len)

print("=== CLASSES PER PATIENT (single-label images only) ===")
print(patient_num_classes.value_counts().sort_index().to_string())

multi_class_patients = patient_num_classes[patient_num_classes > 1]
print(f"\n  Patients appearing in ONLY 1 class : {(patient_num_classes == 1).sum():,}")
print(f"  Patients appearing in 2+ classes   : {len(multi_class_patients):,}")
print(f"  (These are the tricky cases for patient-level splitting)")

if len(multi_class_patients) > 0:
    # Show a few examples
    print("\n=== SAMPLE MULTI-CLASS PATIENTS ===")
    sample_ids = multi_class_patients.head(5).index
    for pid in sample_ids:
        rows = single_label_df[single_label_df['Patient ID'] == pid][['Image Index', 'Finding Labels', 'Follow-up #']]
        print(f"\n  Patient {pid}:")
        print(rows.to_string(index=False))

=== CLASSES PER PATIENT (single-label images only) ===
Finding Labels
1     21577
2      4613
3      1818
4       860
5       370
6       200
7        77
8        31
9        14
10        2
11        2

  Patients appearing in ONLY 1 class : 21,577
  Patients appearing in 2+ classes   : 7,987
  (These are the tricky cases for patient-level splitting)

=== SAMPLE MULTI-CLASS PATIENTS ===

  Patient 5:
     Image Index Finding Labels  Follow-up #
00000005_000.png     No Finding            0
00000005_001.png     No Finding            1
00000005_002.png     No Finding            2
00000005_003.png     No Finding            3
00000005_004.png     No Finding            4
00000005_005.png     No Finding            5
00000005_006.png   Infiltration            6

  Patient 8:
     Image Index Finding Labels  Follow-up #
00000008_000.png   Cardiomegaly            0
00000008_001.png     No Finding            1
00000008_002.png         Nodule            2

  Patient 11:
     Image Index Finding La

In [7]:
single_label_df = df[df['num_labels'] == 1].copy()

print("=== AGE STATS PER CLASS (single-label images) ===")
age_stats = single_label_df.groupby('Finding Labels')['Patient Age'].agg(
    ['count', 'mean', 'median', 'std', 'min', 'max']
).round(1).sort_values('count', ascending=False)
print(age_stats.to_string())

=== AGE STATS PER CLASS (single-label images) ===
                    count  mean  median   std  min  max
Finding Labels                                         
No Finding          60361  45.8    47.0  16.7    1  413
Infiltration         9547  45.1    47.0  17.0    1  151
Atelectasis          4215  50.9    53.0  15.1    2  154
Effusion             3955  50.8    53.0  16.0    5   90
Nodule               2705  50.2    52.0  14.7    2   93
Pneumothorax         2194  45.8    47.0  17.0    5  152
Mass                 2139  48.2    51.0  15.8    2   88
Consolidation        1310  45.2    46.0  17.9    2   89
Pleural_Thickening   1126  50.8    53.0  16.1    1   90
Cardiomegaly         1093  46.7    48.0  17.3    3   91
Emphysema             892  52.4    56.0  18.2   10   88
Fibrosis              727  52.9    54.0  15.1   14   90
Edema                 628  45.1    45.0  21.3    4  414
Pneumonia             322  41.3    42.5  18.3    3   87
Hernia                110  62.0    61.0  13.8   16   8

## Clean up Dataframe

In [8]:
# Rename malformed column names
df = df.rename(columns={
    'OriginalImage[Width': 'img_width',
    'Height]': 'img_height',
    'OriginalImagePixelSpacing[x': 'pixel_spacing_x',
    'y]': 'pixel_spacing_y'
})

# Drop the fully-null column
df = df.drop(columns=['Unnamed: 11'])

# Standardize column names to snake_case
df = df.rename(columns={
    'Image Index': 'image_index',
    'Finding Labels': 'finding_labels',
    'Follow-up #': 'followup_num',
    'Patient ID': 'patient_id',
    'Patient Age': 'patient_age',
    'Patient Gender': 'patient_gender',
    'View Position': 'view_position'
})

# Flag and remove age outliers (>100)
age_outliers = df[df['patient_age'] > 100]
print(f"=== AGE OUTLIERS (>100) ===")
print(age_outliers[['image_index', 'patient_id', 'patient_age', 'finding_labels']].to_string())
print(f"\nDropping {len(age_outliers)} rows with age > 100")
df = df[df['patient_age'] <= 100].reset_index(drop=True)

print(f"\n=== CLEANED DATAFRAME SHAPE ===")
print(df.shape)
print("\n=== COLUMNS ===")
print(df.columns.tolist())
print("\n=== SAMPLE ROW ===")
print(df.iloc[0])

=== AGE OUTLIERS (>100) ===
             image_index  patient_id  patient_age           finding_labels
20852   00005567_000.png        5567          412       Effusion|Pneumonia
46965   00011973_002.png       11973          414                    Edema
48284   00012238_010.png       12238          148               No Finding
55742   00013950_000.png       13950          148               No Finding
58650   00014520_026.png       14520          150        Infiltration|Mass
62929   00015558_000.png       15558          149               No Finding
74884   00018366_044.png       18366          152             Pneumothorax
78795   00019346_000.png       19346          151             Infiltration
84810   00020900_002.png       20900          411               No Finding
85404   00021047_002.png       21047          412  Mass|Pleural_Thickening
86264   00021275_003.png       21275          413               No Finding
91369   00022811_000.png       22811          412               No Findi

In [9]:
# Step 1: single-label only
single_df = df[df['num_labels'] == 1].copy().reset_index(drop=True)
print(f"Single-label images: {len(single_df):,}")

# Step 2: define kept classes (threshold >=1000, + Emphysema at 892)
KEEP_CLASSES = [
    'No Finding',        # will be subsampled later
    'Infiltration',      # 9,547
    'Atelectasis',       # 4,215
    'Effusion',          # 3,955
    'Nodule',            # 2,705
    'Pneumothorax',      # 2,194
    'Mass',              # 2,139
    'Consolidation',     # 1,310
    'Pleural_Thickening',# 1,126
    'Cardiomegaly',      # 1,093
    'Emphysema',         #   892
]

filtered_df = single_df[single_df['finding_labels'].isin(KEEP_CLASSES)].copy().reset_index(drop=True)

print(f"After class filtering: {len(filtered_df):,} images")
print(f"Unique patients remaining: {filtered_df['patient_id'].nunique():,}")

# Step 3: assign integer class label
class_to_idx = {c: i for i, c in enumerate(KEEP_CLASSES)}
filtered_df['label'] = filtered_df['finding_labels'].map(class_to_idx)

print("\n=== CLASS COUNTS AFTER FILTERING ===")
class_summary = filtered_df.groupby(['finding_labels', 'label']).size().reset_index(name='count')
class_summary = class_summary.sort_values('count', ascending=False)
print(class_summary.to_string(index=False))

print("\n=== CLASS INDEX MAPPING ===")
for cls, idx in class_to_idx.items():
    print(f"  {idx:2d} : {cls}")

Single-label images: 91,312
After class filtering: 89,526 images
Unique patients remaining: 29,286

=== CLASS COUNTS AFTER FILTERING ===
    finding_labels  label  count
        No Finding      0  60353
      Infiltration      1   9546
       Atelectasis      2   4214
          Effusion      3   3955
            Nodule      4   2705
      Pneumothorax      5   2193
              Mass      6   2139
     Consolidation      7   1310
Pleural_Thickening      8   1126
      Cardiomegaly      9   1093
         Emphysema     10    892

=== CLASS INDEX MAPPING ===
   0 : No Finding
   1 : Infiltration
   2 : Atelectasis
   3 : Effusion
   4 : Nodule
   5 : Pneumothorax
   6 : Mass
   7 : Consolidation
   8 : Pleural_Thickening
   9 : Cardiomegaly
  10 : Emphysema


In [10]:
import random
random.seed(42)
np.random.seed(42)

TARGET_NO_FINDING = 6000

# Separate No Finding from disease classes
no_finding_df = filtered_df[filtered_df['label'] == 0].copy()
disease_df    = filtered_df[filtered_df['label'] != 0].copy()

print(f"No Finding images   : {len(no_finding_df):,}")
print(f"No Finding patients : {no_finding_df['patient_id'].nunique():,}")
print(f"Disease images      : {len(disease_df):,}")
print(f"Disease patients    : {disease_df['patient_id'].nunique():,}")

# --- Patient-level sampling ---
# We sample PATIENTS first, then take all their No Finding images,
# stopping once we reach TARGET_NO_FINDING images.
# This preserves patient integrity (all images of a patient stay together).

nf_patients = no_finding_df['patient_id'].unique().tolist()
random.shuffle(nf_patients)

selected_patients = []
selected_count = 0

for pid in nf_patients:
    if selected_count >= TARGET_NO_FINDING:
        break
    n = (no_finding_df['patient_id'] == pid).sum()
    selected_patients.append(pid)
    selected_count += n

nf_sampled_df = no_finding_df[no_finding_df['patient_id'].isin(selected_patients)].copy()

print(f"\n=== NO FINDING AFTER PATIENT-LEVEL SAMPLING ===")
print(f"  Selected patients : {len(selected_patients):,}")
print(f"  Selected images   : {len(nf_sampled_df):,}  (target was {TARGET_NO_FINDING:,})")

# --- Combine back ---
final_df = pd.concat([nf_sampled_df, disease_df], ignore_index=True)
final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\n=== FINAL DATASET ===")
print(f"  Total images   : {len(final_df):,}")
print(f"  Total patients : {final_df['patient_id'].nunique():,}")

print("\n=== FINAL CLASS DISTRIBUTION ===")
final_counts = final_df.groupby(['finding_labels', 'label']).size().reset_index(name='count')
final_counts = final_counts.sort_values('count', ascending=False)
final_counts['pct'] = (final_counts['count'] / len(final_df) * 100).round(2)
print(final_counts.to_string(index=False))

print("\n=== GENDER DISTRIBUTION IN FINAL SET ===")
print(final_df['patient_gender'].value_counts())
print(final_df['patient_gender'].value_counts(normalize=True).round(3))

No Finding images   : 60,353
No Finding patients : 24,905
Disease images      : 29,173
Disease patients    : 11,752

=== NO FINDING AFTER PATIENT-LEVEL SAMPLING ===
  Selected patients : 2,456
  Selected images   : 6,008  (target was 6,000)

=== FINAL DATASET ===
  Total images   : 35,181
  Total patients : 13,481

=== FINAL CLASS DISTRIBUTION ===
    finding_labels  label  count   pct
      Infiltration      1   9546 27.13
        No Finding      0   6008 17.08
       Atelectasis      2   4214 11.98
          Effusion      3   3955 11.24
            Nodule      4   2705  7.69
      Pneumothorax      5   2193  6.23
              Mass      6   2139  6.08
     Consolidation      7   1310  3.72
Pleural_Thickening      8   1126  3.20
      Cardiomegaly      9   1093  3.11
         Emphysema     10    892  2.54

=== GENDER DISTRIBUTION IN FINAL SET ===
patient_gender
M    19500
F    15681
Name: count, dtype: int64
patient_gender
M    0.554
F    0.446
Name: proportion, dtype: float64


In [11]:
from sklearn.model_selection import train_test_split

# ── Strategy ──────────────────────────────────────────────────────────────────
# Unit of splitting is the PATIENT, not the image.
# For patients with images in multiple classes we assign a single
# "dominant class" (most frequent label for that patient) so that
# stratification still works sensibly.
# All images of a patient land in exactly one split → no leakage.
# Split ratio: 70 / 10 / 20  (train / val / test)
# ──────────────────────────────────────────────────────────────────────────────

# Build one row per patient with their dominant label
patient_meta = (
    final_df.groupby('patient_id')['label']
    .agg(lambda x: x.value_counts().idxmax())   # dominant class
    .reset_index()
    .rename(columns={'label': 'dominant_label'})
)

print(f"Total patients to split: {len(patient_meta):,}")
print("\n=== DOMINANT LABEL DISTRIBUTION ACROSS PATIENTS ===")
dom_counts = patient_meta['dominant_label'].value_counts().sort_index()
for idx, cnt in dom_counts.items():
    cls_name = KEEP_CLASSES[idx]
    print(f"  {idx:2d} {cls_name:<20} {cnt:>5,} patients")

# ── First cut: train  vs  (val + test) ────────────────────────────────────────
train_patients, valtest_patients = train_test_split(
    patient_meta,
    test_size=0.30,
    stratify=patient_meta['dominant_label'],
    random_state=42
)

# ── Second cut: val  vs  test  (split the 30% evenly → 10% / 20%) ─────────────
val_patients, test_patients = train_test_split(
    valtest_patients,
    test_size=0.667,          # 0.667 × 30% ≈ 20% of total
    stratify=valtest_patients['dominant_label'],
    random_state=42
)

print(f"\n=== PATIENT COUNTS PER SPLIT ===")
print(f"  Train : {len(train_patients):,}")
print(f"  Val   : {len(val_patients):,}")
print(f"  Test  : {len(test_patients):,}")
print(f"  Total : {len(train_patients) + len(val_patients) + len(test_patients):,}")

# ── Map patients → images ──────────────────────────────────────────────────────
train_ids = set(train_patients['patient_id'])
val_ids   = set(val_patients['patient_id'])
test_ids  = set(test_patients['patient_id'])

train_df = final_df[final_df['patient_id'].isin(train_ids)].copy().reset_index(drop=True)
val_df   = final_df[final_df['patient_id'].isin(val_ids)].copy().reset_index(drop=True)
test_df  = final_df[final_df['patient_id'].isin(test_ids)].copy().reset_index(drop=True)

# ── Sanity checks ──────────────────────────────────────────────────────────────
assert len(set(train_df['patient_id']) & set(val_df['patient_id']))  == 0, "LEAKAGE: train/val"
assert len(set(train_df['patient_id']) & set(test_df['patient_id'])) == 0, "LEAKAGE: train/test"
assert len(set(val_df['patient_id'])   & set(test_df['patient_id'])) == 0, "LEAKAGE: val/test"
print("\n✓ No patient overlap between splits")

print(f"\n=== IMAGE COUNTS PER SPLIT ===")
print(f"  Train : {len(train_df):,}  ({len(train_df)/len(final_df)*100:.1f}%)")
print(f"  Val   : {len(val_df):,}   ({len(val_df)/len(final_df)*100:.1f}%)")
print(f"  Test  : {len(test_df):,}  ({len(test_df)/len(final_df)*100:.1f}%)")

print(f"\n=== CLASS DISTRIBUTION PER SPLIT (image counts) ===")
for split_name, split in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    counts = split['label'].value_counts().sort_index()
    print(f"\n  {split_name}:")
    for idx, cnt in counts.items():
        print(f"    {KEEP_CLASSES[idx]:<22} {cnt:>5,}  ({cnt/len(split)*100:.1f}%)")

print(f"\n=== GENDER DISTRIBUTION PER SPLIT ===")
for split_name, split in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    g = split['patient_gender'].value_counts(normalize=True).round(3)
    print(f"  {split_name}: M={g.get('M', 0):.3f}  F={g.get('F', 0):.3f}")

Total patients to split: 13,481

=== DOMINANT LABEL DISTRIBUTION ACROSS PATIENTS ===
   0 No Finding           2,268 patients
   1 Infiltration         4,151 patients
   2 Atelectasis          1,687 patients
   3 Effusion             1,125 patients
   4 Nodule               1,229 patients
   5 Pneumothorax           506 patients
   6 Mass                   832 patients
   7 Consolidation          279 patients
   8 Pleural_Thickening     508 patients
   9 Cardiomegaly           583 patients
  10 Emphysema              313 patients

=== PATIENT COUNTS PER SPLIT ===
  Train : 9,436
  Val   : 1,346
  Test  : 2,699
  Total : 13,481

✓ No patient overlap between splits

=== IMAGE COUNTS PER SPLIT ===
  Train : 24,654  (70.1%)
  Val   : 3,725   (10.6%)
  Test  : 6,802  (19.3%)

=== CLASS DISTRIBUTION PER SPLIT (image counts) ===

  Train:
    No Finding             4,276  (17.3%)
    Infiltration           6,661  (27.0%)
    Atelectasis            2,955  (12.0%)
    Effusion               2,7

In [14]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
import os

os.makedirs('./data', exist_ok=True)

CLASS_NAMES  = KEEP_CLASSES
SHORT_NAMES  = ['NoFind', 'Infilt', 'Atelec', 'Effus', 'Nodule',
                'PneuTx', 'Mass', 'Consol', 'PlThck', 'CardMeg', 'Emphy']
SPLIT_COLORS = {'Train': '#4C72B0', 'Val': '#DD8452', 'Test': '#55A868'}
GENDER_COLORS = {'M': '#5B8DB8', 'F': '#C0616B'}

fig = plt.figure(figsize=(24, 32))
gs  = gridspec.GridSpec(4, 3, figure=fig, hspace=0.5, wspace=0.38)

# ── 1. Horizontal bar — total image counts per class (full final_df) ──────────
ax1 = fig.add_subplot(gs[0, 0])
counts = [final_df[final_df['label'] == j].shape[0] for j in range(len(CLASS_NAMES))]
sorted_idx = np.argsort(counts)
colors_bar = plt.cm.RdYlGn(np.linspace(0.25, 0.85, len(CLASS_NAMES)))
bars = ax1.barh(
    [SHORT_NAMES[i] for i in sorted_idx],
    [counts[i] for i in sorted_idx],
    color=[colors_bar[i] for i in sorted_idx],
    edgecolor='white', height=0.7
)
for bar, val in zip(bars, [counts[i] for i in sorted_idx]):
    ax1.text(bar.get_width() + 30, bar.get_y() + bar.get_height()/2,
             f'{val:,}', va='center', fontsize=8)
ax1.set_xlabel('Image Count')
ax1.set_title('Images per Class\n(filtered subset)', fontweight='bold')
ax1.grid(axis='x', alpha=0.3)
ax1.set_xlim(0, max(counts) * 1.18)

# ── 2. Pie chart — overall class share ────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
wedge_colors = plt.cm.tab20.colors[:len(CLASS_NAMES)]
wedges, texts, autotexts = ax2.pie(
    counts,
    labels=SHORT_NAMES,
    autopct=lambda p: f'{p:.1f}%' if p > 2.5 else '',
    colors=wedge_colors,
    startangle=140,
    pctdistance=0.78,
    wedgeprops=dict(edgecolor='white', linewidth=1.2)
)
for t in texts:    t.set_fontsize(8)
for t in autotexts: t.set_fontsize(7.5)
ax2.set_title('Class Share — Full Filtered Set\n(35,181 images)', fontweight='bold')

# ── 3. Stacked 100% bar — class proportion per split ─────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
split_names = ['Train', 'Val', 'Test']
split_dfs   = [train_df, val_df, test_df]
proportions = np.array([
    [s[s['label'] == j].shape[0] / len(s) * 100 for j in range(len(CLASS_NAMES))]
    for s in split_dfs
])
bottom = np.zeros(3)
for j in range(len(CLASS_NAMES)):
    ax3.bar(split_names, proportions[:, j], bottom=bottom,
            color=wedge_colors[j], edgecolor='white', linewidth=0.5, label=SHORT_NAMES[j])
    bottom += proportions[:, j]
ax3.set_ylabel('% of Split')
ax3.set_title('Class Proportion\nper Split (stacked 100%)', fontweight='bold')
ax3.legend(loc='upper right', fontsize=6.5, ncol=2, bbox_to_anchor=(1.42, 1.02))
ax3.set_ylim(0, 100)
ax3.grid(axis='y', alpha=0.3)

# ── 4. Violin plot — age distribution per class ───────────────────────────────
ax4 = fig.add_subplot(gs[1, :2])
age_data = [final_df[final_df['label'] == j]['patient_age'].values
            for j in range(len(CLASS_NAMES))]
parts = ax4.violinplot(age_data, positions=range(len(CLASS_NAMES)),
                       showmedians=True, showextrema=True, widths=0.7)
for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(wedge_colors[i])
    pc.set_alpha(0.75)
parts['cmedians'].set_color('black')
parts['cmedians'].set_linewidth(2)
parts['cbars'].set_color('gray')
parts['cmins'].set_color('gray')
parts['cmaxes'].set_color('gray')
ax4.set_xticks(range(len(CLASS_NAMES)))
ax4.set_xticklabels(SHORT_NAMES, fontsize=9)
ax4.set_ylabel('Patient Age')
ax4.set_title('Age Distribution per Class (Violin)', fontweight='bold')
ax4.grid(axis='y', alpha=0.3)

# ── 5. Heatmap — gender × class counts ───────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
gender_matrix = np.array([
    [final_df[(final_df['label'] == j) & (final_df['patient_gender'] == g)].shape[0]
     for g in ['M', 'F']]
    for j in range(len(CLASS_NAMES))
])
sns.heatmap(
    gender_matrix,
    annot=True, fmt='d', cmap='YlOrRd',
    xticklabels=['Male', 'Female'],
    yticklabels=SHORT_NAMES,
    linewidths=0.5, ax=ax5,
    cbar_kws={'label': 'Image count'}
)
ax5.set_title('Gender × Class\nImage Counts (heatmap)', fontweight='bold')
ax5.tick_params(axis='y', labelsize=8.5)

# ── 6. Grouped bar — gender % per class (M vs F) ─────────────────────────────
ax6 = fig.add_subplot(gs[2, :2])
totals  = gender_matrix.sum(axis=1)
m_pcts  = gender_matrix[:, 0] / totals * 100
f_pcts  = gender_matrix[:, 1] / totals * 100
x6      = np.arange(len(CLASS_NAMES))
width6  = 0.38
ax6.bar(x6 - width6/2, m_pcts, width6, label='Male',
        color=GENDER_COLORS['M'], alpha=0.85, edgecolor='white')
ax6.bar(x6 + width6/2, f_pcts, width6, label='Female',
        color=GENDER_COLORS['F'], alpha=0.85, edgecolor='white')
ax6.axhline(50, color='black', linestyle='--', linewidth=1, alpha=0.5, label='50% line')
ax6.set_xticks(x6)
ax6.set_xticklabels(SHORT_NAMES, fontsize=9)
ax6.set_ylabel('% within class')
ax6.set_ylim(0, 80)
ax6.set_title('Gender Split per Class (%)', fontweight='bold')
ax6.legend(fontsize=9)
ax6.grid(axis='y', alpha=0.3)

# ── 7. KDE — age distribution by gender (overall) ────────────────────────────
ax7 = fig.add_subplot(gs[2, 2])
for gender, color in GENDER_COLORS.items():
    ages = final_df[final_df['patient_gender'] == gender]['patient_age']
    ages.plot.kde(ax=ax7, label=f'{gender} (n={len(ages):,})',
                  color=color, linewidth=2.5)
ax7.set_xlabel('Patient Age')
ax7.set_ylabel('Density')
ax7.set_title('Age Density by Gender\n(full filtered set)', fontweight='bold')
ax7.legend(fontsize=9)
ax7.grid(alpha=0.3)
ax7.set_xlim(0, 100)

# ── 8. Scatter — images per patient vs followup number, colored by dominant class
ax8 = fig.add_subplot(gs[3, :2])
pat_summary = final_df.groupby('patient_id').agg(
    n_images=('image_index', 'count'),
    max_followup=('followup_num', 'max'),
    dominant_label=('label', lambda x: x.value_counts().idxmax())
).reset_index()

for j in range(len(CLASS_NAMES)):
    sub = pat_summary[pat_summary['dominant_label'] == j]
    ax8.scatter(sub['max_followup'], sub['n_images'],
                alpha=0.35, s=18, color=wedge_colors[j],
                label=SHORT_NAMES[j])
ax8.set_xlabel('Max Follow-up Number')
ax8.set_ylabel('Total Images for Patient')
ax8.set_title('Images per Patient vs Max Follow-up\n(colored by dominant class)', fontweight='bold')
ax8.legend(fontsize=7, ncol=3, loc='upper left')
ax8.grid(alpha=0.3)

# ── 9. Donut — train/val/test image split ────────────────────────────────────
ax9 = fig.add_subplot(gs[3, 2])
split_counts = [len(train_df), len(val_df), len(test_df)]
split_labels = [f'Train\n{len(train_df):,}\n({len(train_df)/len(final_df)*100:.1f}%)',
                f'Val\n{len(val_df):,}\n({len(val_df)/len(final_df)*100:.1f}%)',
                f'Test\n{len(test_df):,}\n({len(test_df)/len(final_df)*100:.1f}%)']
wedges9, _ = ax9.pie(
    split_counts,
    labels=split_labels,
    colors=[SPLIT_COLORS['Train'], SPLIT_COLORS['Val'], SPLIT_COLORS['Test']],
    startangle=90,
    wedgeprops=dict(width=0.5, edgecolor='white', linewidth=2),
    textprops={'fontsize': 9}
)
ax9.set_title('Train / Val / Test\nImage Split (donut)', fontweight='bold')
centre_text = f'Total\n{len(final_df):,}'
ax9.text(0, 0, centre_text, ha='center', va='center', fontsize=10, fontweight='bold')

plt.suptitle(
    'NIH ChestX-ray14 — Filtered Subset Deep Dive\n'
    '35,181 images · 13,481 patients · 11 classes',
    fontsize=15, fontweight='bold', y=1.01
)

plt.savefig('./data/dataset_overview_v2.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → ./data/dataset_overview_v2.png")

Saved → ./data/dataset_overview_v2.png


## Download Data

In [15]:
import os

SAVE_DIR = './data'
os.makedirs(SAVE_DIR, exist_ok=True)

# Save each split as a CSV — these are the source of truth for all downstream work
train_df.to_csv(f'{SAVE_DIR}/train_split.csv', index=False)
val_df.to_csv(f'{SAVE_DIR}/val_split.csv',     index=False)
test_df.to_csv(f'{SAVE_DIR}/test_split.csv',   index=False)
final_df.to_csv(f'{SAVE_DIR}/final_filtered.csv', index=False)

# Also save the class index mapping for reference
import json
with open(f'{SAVE_DIR}/class_to_idx.json', 'w') as f:
    json.dump(class_to_idx, f, indent=2)

print("=== SAVED FILES ===")
for fname in ['train_split.csv', 'val_split.csv', 'test_split.csv',
              'final_filtered.csv', 'class_to_idx.json']:
    fpath = f'{SAVE_DIR}/{fname}'
    size  = os.path.getsize(fpath) / 1024
    print(f"  {fname:<25} {size:>8.1f} KB")

# Build the master list of image filenames we actually need to download
image_filenames = final_df['image_index'].unique().tolist()
print(f"\n=== UNIQUE IMAGES TO DOWNLOAD ===")
print(f"  {len(image_filenames):,} images (out of 112,120 original)")

# Save this list — we'll use it to filter the kaggle download
with open(f'{SAVE_DIR}/images_to_download.txt', 'w') as f:
    for fname in sorted(image_filenames):
        f.write(fname + '\n')

print(f"  Saved → {SAVE_DIR}/images_to_download.txt")

# Quick sanity: confirm all three splits are covered
all_in_final = set(final_df['image_index'])
all_in_splits = set(train_df['image_index']) | set(val_df['image_index']) | set(test_df['image_index'])
assert all_in_final == all_in_splits, "MISMATCH: some images missing from splits"
print("\n✓ All images in final_df are accounted for across splits")
print(f"✓ Class mapping saved: {class_to_idx}")

=== SAVED FILES ===
  train_split.csv             1698.6 KB
  val_split.csv                257.2 KB
  test_split.csv               469.4 KB
  final_filtered.csv          2424.8 KB
  class_to_idx.json              0.2 KB

=== UNIQUE IMAGES TO DOWNLOAD ===
  35,181 images (out of 112,120 original)
  Saved → ./data/images_to_download.txt

✓ All images in final_df are accounted for across splits
✓ Class mapping saved: {'No Finding': 0, 'Infiltration': 1, 'Atelectasis': 2, 'Effusion': 3, 'Nodule': 4, 'Pneumothorax': 5, 'Mass': 6, 'Consolidation': 7, 'Pleural_Thickening': 8, 'Cardiomegaly': 9, 'Emphysema': 10}


In [16]:
import subprocess
import os

# Check kaggle is installed
result = subprocess.run(['kaggle', '--version'], capture_output=True, text=True)
print("=== KAGGLE VERSION ===")
print(result.stdout.strip() if result.returncode == 0 else f"ERROR: {result.stderr.strip()}")

# Check credentials file exists
cred_path = os.path.expanduser('~/.kaggle/kaggle.json')
if os.path.exists(cred_path):
    # Check permissions (should be 600)
    perms = oct(os.stat(cred_path).st_mode)[-3:]
    print(f"\n=== KAGGLE CREDENTIALS ===")
    print(f"  Found   : {cred_path}")
    print(f"  Perms   : {perms}  {'✓' if perms == '600' else '⚠ should be 600'}")
else:
    print(f"\n✗ No credentials found at {cred_path}")
    print("  To fix:")
    print("  1. Go to https://www.kaggle.com/settings → API → Create New Token")
    print("  2. This downloads kaggle.json")
    print("  3. Run in terminal:")
    print("       mkdir -p ~/.kaggle")
    print("       mv /path/to/kaggle.json ~/.kaggle/kaggle.json")
    print("       chmod 600 ~/.kaggle/kaggle.json")

# Check GCS bucket access while we're at it
print("\n=== GCS BUCKET ACCESS CHECK ===")
result_gcs = subprocess.run(
    ['gsutil', 'ls', 'gs://'],
    capture_output=True, text=True
)
if result_gcs.returncode == 0:
    print("  gsutil available ✓")
else:
    print(f"  gsutil check: {result_gcs.stderr.strip()[:200]}")

# Check available disk space
result_disk = subprocess.run(['df', '-h', '.'], capture_output=True, text=True)
print("\n=== DISK SPACE ===")
print(result_disk.stdout.strip())

=== KAGGLE VERSION ===
Kaggle API 1.7.4.5

=== KAGGLE CREDENTIALS ===
  Found   : /home/jupyter/.kaggle/kaggle.json
  Perms   : 600  ✓

=== GCS BUCKET ACCESS CHECK ===
  gsutil available ✓

=== DISK SPACE ===
Filesystem      Size  Used Avail Use% Mounted on
/dev/nvme0n2    2.0T  919G  1.1T  47% /home/jupyter


In [3]:
import kagglehub
import os

print("=== TESTING KAGGLEHUB DOWNLOAD ===")
print("Note: first run will download, subsequent runs use cache\n")

path = kagglehub.dataset_download("nih-chest-xrays/data")

print(f"\nPath to dataset files: {path}")

# Inspect what landed
print("\n=== TOP LEVEL CONTENTS ===")
for item in sorted(os.listdir(path)):
    full = os.path.join(path, item)
    if os.path.isdir(full):
        n_files = sum(len(f) for _, _, f in os.walk(full))
        print(f"  [DIR]  {item}/  ({n_files} files)")
    else:
        size_mb = os.path.getsize(full) / 1e6
        print(f"  [FILE] {item}  ({size_mb:.1f} MB)")

# Check one image folder structure
print("\n=== SAMPLE FOLDER STRUCTURE (images_001) ===")
img_001 = os.path.join(path, 'images_001')
if os.path.exists(img_001):
    for item in sorted(os.listdir(img_001))[:5]:
        print(f"  {item}")
    sub = os.path.join(img_001, 'images')
    if os.path.exists(sub):
        sample_files = sorted(os.listdir(sub))[:5]
        print(f"  images/")
        for f in sample_files:
            print(f"    {f}")
else:
    print(f"  images_001 not found at {img_001}")
    print("  Full directory tree (2 levels):")
    for root, dirs, files in os.walk(path):
        level = root.replace(path, '').count(os.sep)
        if level > 2:
            continue
        indent = '  ' * level
        print(f'{indent}{os.path.basename(root)}/')
        if level == 2:
            print(f'{indent}  ({len(files)} files)')

/opt/conda/envs/re_id_research_v1/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== TESTING KAGGLEHUB DOWNLOAD ===
Note: first run will download, subsequent runs use cache



100%|██████████| 42.0G/42.0G [04:57<00:00, 152MB/s] 


Extracting files...

Path to dataset files: /home/jupyter/.cache/kagglehub/datasets/nih-chest-xrays/data/versions/3

=== TOP LEVEL CONTENTS ===
  [FILE] ARXIV_V5_CHESTXRAY.pdf  (9.0 MB)
  [FILE] BBox_List_2017.csv  (0.1 MB)
  [FILE] Data_Entry_2017.csv  (7.9 MB)
  [FILE] FAQ_CHESTXRAY.pdf  (0.1 MB)
  [FILE] LOG_CHESTXRAY.pdf  (0.0 MB)
  [FILE] README_CHESTXRAY.pdf  (0.8 MB)
  [DIR]  images_001/  (4999 files)
  [DIR]  images_002/  (10000 files)
  [DIR]  images_003/  (10000 files)
  [DIR]  images_004/  (10000 files)
  [DIR]  images_005/  (10000 files)
  [DIR]  images_006/  (10000 files)
  [DIR]  images_007/  (10000 files)
  [DIR]  images_008/  (10000 files)
  [DIR]  images_009/  (10000 files)
  [DIR]  images_010/  (10000 files)
  [DIR]  images_011/  (10000 files)
  [DIR]  images_012/  (7121 files)
  [FILE] test_list.txt  (0.4 MB)
  [FILE] train_val_list.txt  (1.5 MB)

=== SAMPLE FOLDER STRUCTURE (images_001) ===
  images
  images/
    00000001_000.png
    00000001_001.png
    00000001_00

In [5]:
import os
import shutil
import subprocess
from pathlib import Path

# ── CONFIG ────────────────────────────────────────────────────────────────────
GCS_IMG_DIR = 'gs://nih-cxr-say/nih_cxr/images'
LOCAL_IMG_DIR = './data/chest/images'
os.makedirs(LOCAL_IMG_DIR, exist_ok=True)

# The path returned by kagglehub.dataset_download()
# Paste the actual path here once download completes
CACHE_PATH = path  # this is already set from Cell 23

# Load our required image list
with open('./data/metadata/images_to_download.txt', 'r') as f:
    needed_images = set(line.strip() for line in f)

print(f"Images needed       : {len(needed_images):,}")
print(f"Cache path          : {CACHE_PATH}")
print(f"Local staging dir   : {LOCAL_IMG_DIR}")
print(f"GCS target          : {GCS_IMG_DIR}")

# ── Step 1: Build index of all PNGs in cache ──────────────────────────────────
print("\n=== BUILDING CACHE INDEX ===")
cache_index = {}  # filename → full path in cache

for root, dirs, files in os.walk(CACHE_PATH):
    for fname in files:
        if fname.endswith('.png'):
            cache_index[fname] = os.path.join(root, fname)

print(f"Total PNGs in cache : {len(cache_index):,}")

# Check overlap
found     = needed_images & set(cache_index.keys())
not_found = needed_images - set(cache_index.keys())
print(f"Needed & found      : {len(found):,}")
print(f"Needed & NOT found  : {len(not_found):,}")

if not_found:
    print(f"\n  ⚠ Missing (first 10):")
    for f in sorted(not_found)[:10]:
        print(f"    {f}")

# ── Step 2: Copy needed images to flat local staging dir ─────────────────────
print("\n=== COPYING FILTERED IMAGES TO LOCAL STAGING ===")
copied  = 0
skipped = 0
errors  = []

for fname in sorted(found):
    src  = cache_index[fname]
    dest = os.path.join(LOCAL_IMG_DIR, fname)
    
    if os.path.exists(dest):
        skipped += 1
        continue
    try:
        shutil.copy2(src, dest)
        copied += 1
    except Exception as e:
        errors.append((fname, str(e)))
    
    if copied % 5000 == 0 and copied > 0:
        print(f"  Copied {copied:,} so far...")

print(f"\n  Copied  : {copied:,}")
print(f"  Skipped : {skipped:,}  (already existed)")
print(f"  Errors  : {len(errors):,}")
if errors:
    for fname, err in errors[:5]:
        print(f"    {fname}: {err}")

# Verify local count
local_pngs = [f for f in os.listdir(LOCAL_IMG_DIR) if f.endswith('.png')]
print(f"\n  Local PNG count : {len(local_pngs):,}  (expected {len(needed_images):,})")

# ── Step 3: Upload to GCS in one parallel batch ───────────────────────────────
print("\n=== UPLOADING TO GCS ===")
result = subprocess.run(
    f'gsutil -m cp {LOCAL_IMG_DIR}/*.png {GCS_IMG_DIR}/',
    shell=True, capture_output=True, text=True
)
print(f"Return code : {result.returncode}")
if result.returncode == 0:
    print("  ✓ Upload complete")
else:
    print(f"  STDERR: {result.stderr.strip()[:500]}")

# ── Step 4: Verify GCS count ──────────────────────────────────────────────────
print("\n=== VERIFYING GCS COUNT ===")
result2 = subprocess.run(
    f'gsutil ls {GCS_IMG_DIR}/*.png | wc -l',
    shell=True, capture_output=True, text=True
)
gcs_count = int(result2.stdout.strip())
print(f"  GCS image count : {gcs_count:,}  (expected {len(needed_images):,})")
if gcs_count == len(needed_images):
    print("  ✓ All images verified in GCS")
else:
    diff = len(needed_images) - gcs_count
    print(f"  ⚠ Discrepancy of {diff:,} images")

# ── Step 5: Clean up local staging to free disk ───────────────────────────────
print("\n=== CLEANING UP LOCAL STAGING ===")
confirm = input("Delete local staging images to free disk? (yes/no): ")
if confirm.strip().lower() == 'yes':
    shutil.rmtree(LOCAL_IMG_DIR)
    os.makedirs(LOCAL_IMG_DIR, exist_ok=True)
    print(f"  ✓ Cleared {LOCAL_IMG_DIR}")
else:
    print("  Skipped — local images retained")

print("\n=== DONE ===")
print(f"  Images in GCS   : {gcs_count:,}")
print(f"  Metadata in GCS : gs://nih-cxr-say/nih_cxr/metadata/")
print(f"  Ready for NPZ pipeline")

Images needed       : 35,181
Cache path          : /home/jupyter/.cache/kagglehub/datasets/nih-chest-xrays/data/versions/3
Local staging dir   : ./data/chest/images
GCS target          : gs://nih-cxr-say/nih_cxr/images

=== BUILDING CACHE INDEX ===
Total PNGs in cache : 112,120
Needed & found      : 35,181
Needed & NOT found  : 0

=== COPYING FILTERED IMAGES TO LOCAL STAGING ===
  Copied 5,000 so far...
  Copied 10,000 so far...
  Copied 15,000 so far...
  Copied 20,000 so far...
  Copied 25,000 so far...
  Copied 30,000 so far...
  Copied 35,000 so far...

  Copied  : 35,181
  Skipped : 0  (already existed)
  Errors  : 0

  Local PNG count : 35,181  (expected 35,181)

=== UPLOADING TO GCS ===
Return code : 0
  ✓ Upload complete

=== VERIFYING GCS COUNT ===
  GCS image count : 35,181  (expected 35,181)
  ✓ All images verified in GCS

=== CLEANING UP LOCAL STAGING ===


Delete local staging images to free disk? (yes/no):  y


  Skipped — local images retained

=== DONE ===
  Images in GCS   : 35,181
  Metadata in GCS : gs://nih-cxr-say/nih_cxr/metadata/
  Ready for NPZ pipeline


In [7]:
import subprocess
import shutil
import os

CACHE_PATH = os.path.expanduser('~/.cache/kagglehub')

# Check size
result = subprocess.run(
    f'du -sh {CACHE_PATH}',
    shell=True, capture_output=True, text=True
)
print(f"=== KAGGLEHUB CACHE SIZE ===")
print(f"  {result.stdout.strip()}")

# Show structure
result2 = subprocess.run(
    f'find {CACHE_PATH} -maxdepth 4 -type d',
    shell=True, capture_output=True, text=True
)
print(f"\n=== CACHE DIRECTORY STRUCTURE ===")
print(result2.stdout.strip())

# Clear it
confirm = input("\nDelete kagglehub cache to free disk? (yes/no): ")
if confirm.strip().lower() == 'yes':
    shutil.rmtree(CACHE_PATH)
    print(f"  ✓ Cleared {CACHE_PATH}")
else:
    print("  Skipped")

# Final disk check
result3 = subprocess.run('df -h .', shell=True, capture_output=True, text=True)
print(f"\n=== DISK SPACE AFTER CLEANUP ===")
print(result3.stdout.strip())

=== KAGGLEHUB CACHE SIZE ===
  43G	/home/jupyter/.cache/kagglehub

=== CACHE DIRECTORY STRUCTURE ===
/home/jupyter/.cache/kagglehub
/home/jupyter/.cache/kagglehub/datasets
/home/jupyter/.cache/kagglehub/datasets/nih-chest-xrays
/home/jupyter/.cache/kagglehub/datasets/nih-chest-xrays/data
/home/jupyter/.cache/kagglehub/datasets/nih-chest-xrays/data/versions



Delete kagglehub cache to free disk? (yes/no):  yes


  ✓ Cleared /home/jupyter/.cache/kagglehub

=== DISK SPACE AFTER CLEANUP ===
Filesystem      Size  Used Avail Use% Mounted on
/dev/nvme0n2    2.0T  919G  1.1T  47% /home/jupyter


In [8]:
import numpy as np
import pandas as pd
from PIL import Image
import subprocess
import os
import io
import json
import time

# ── CONFIG ────────────────────────────────────────────────────────────────────
GCS_IMG_DIR   = 'gs://nih-cxr-say/nih_cxr/images'
LOCAL_NPZ_DIR = './data/chest/npz'
META_DIR      = './data/metadata'
IMG_SIZE      = 224
BATCH_SIZE    = 500   # images pulled from GCS at a time
TMP_DIR       = './data/chest/tmp_imgs'

os.makedirs(LOCAL_NPZ_DIR, exist_ok=True)
os.makedirs(TMP_DIR,       exist_ok=True)

# Load splits and class mapping
train_df = pd.read_csv(f'{META_DIR}/train_split.csv')
val_df   = pd.read_csv(f'{META_DIR}/val_split.csv')
test_df  = pd.read_csv(f'{META_DIR}/test_split.csv')

with open(f'{META_DIR}/class_to_idx.json') as f:
    class_to_idx = json.load(f)

# Encode gender as int: M=1, F=0
for df in [train_df, val_df, test_df]:
    df['gender_int'] = (df['patient_gender'] == 'M').astype(np.int8)

print(f"IMG_SIZE  : {IMG_SIZE}x{IMG_SIZE}")
print(f"Splits    : train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}")
print(f"NPZ dir   : {LOCAL_NPZ_DIR}")
print(f"Storage   : float16 images (space efficient)")

# ── Core function: process one split ─────────────────────────────────────────
def build_npz(split_df, split_name):
    print(f"\n{'='*60}")
    print(f"Building {split_name}.npz  ({len(split_df):,} images)")
    print(f"{'='*60}")

    n          = len(split_df)
    images     = np.zeros((n, IMG_SIZE, IMG_SIZE), dtype=np.float16)
    labels     = np.zeros(n,  dtype=np.int64)
    subject_ids= np.zeros(n,  dtype=np.int64)
    study_ids  = np.zeros(n,  dtype=np.int64)
    genders    = np.zeros(n,  dtype=np.int8)

    failed     = []
    t0         = time.time()

    # Process in batches to avoid pulling too many files at once
    for batch_start in range(0, n, BATCH_SIZE):
        batch_end  = min(batch_start + BATCH_SIZE, n)
        batch      = split_df.iloc[batch_start:batch_end]
        batch_fnames = batch['image_index'].tolist()

        # Pull batch from GCS into tmp dir
        filelist = ' '.join([f'{GCS_IMG_DIR}/{f}' for f in batch_fnames])
        result   = subprocess.run(
            f'gsutil -m cp {filelist} {TMP_DIR}/',
            shell=True, capture_output=True, text=True
        )
        if result.returncode != 0:
            print(f"  ⚠ GCS batch pull error: {result.stderr.strip()[:200]}")

        # Process each image in batch
        for local_i, (_, row) in enumerate(batch.iterrows()):
            global_i = batch_start + local_i
            fpath    = os.path.join(TMP_DIR, row['image_index'])

            try:
                img = Image.open(fpath).convert('L')          # grayscale
                img = img.resize((IMG_SIZE, IMG_SIZE),
                                 Image.LANCZOS)                # 224x224
                arr = np.array(img, dtype=np.float32) / 255.0 # normalize 0-1
                images[global_i]      = arr.astype(np.float16)
                labels[global_i]      = row['label']
                subject_ids[global_i] = row['patient_id']
                study_ids[global_i]   = row['followup_num']
                genders[global_i]     = row['gender_int']
            except Exception as e:
                failed.append((row['image_index'], str(e)))
                print(f"  ✗ Failed: {row['image_index']} — {e}")

            # Clean up tmp file immediately
            if os.path.exists(fpath):
                os.remove(fpath)

        # Progress report
        elapsed = time.time() - t0
        rate    = (batch_end) / elapsed
        eta     = (n - batch_end) / rate if rate > 0 else 0
        print(f"  [{batch_end:>6,}/{n:,}]  "
              f"elapsed: {elapsed/60:.1f}m  "
              f"rate: {rate:.0f} img/s  "
              f"ETA: {eta/60:.1f}m")

    # Save NPZ
    out_path = f'{LOCAL_NPZ_DIR}/{split_name}.npz'
    np.savez_compressed(
        out_path,
        images      = images,
        labels      = labels,
        subject_ids = subject_ids,
        study_ids   = study_ids,
        genders     = genders
    )

    size_gb = os.path.getsize(out_path) / 1e9
    elapsed = (time.time() - t0) / 60

    print(f"\n  ✓ Saved  : {out_path}")
    print(f"  Size     : {size_gb:.2f} GB")
    print(f"  Failed   : {len(failed)}")
    print(f"  Time     : {elapsed:.1f} min")

    if failed:
        print(f"  Failed images (first 5):")
        for fname, err in failed[:5]:
            print(f"    {fname}: {err}")

    return failed

# ── Run all three splits ───────────────────────────────────────────────────────
all_failed = {}
for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    failed = build_npz(df, name)
    all_failed[name] = failed

# ── Final summary ──────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"ALL SPLITS COMPLETE")
print(f"{'='*60}")
total_size = 0
for name in ['train', 'val', 'test']:
    p    = f'{LOCAL_NPZ_DIR}/{name}.npz'
    size = os.path.getsize(p) / 1e9
    total_size += size
    print(f"  {name}.npz  :  {size:.2f} GB  |  failed: {len(all_failed[name])}")

print(f"\n  Total NPZ size : {total_size:.2f} GB")
print(f"  Failed images  : {sum(len(v) for v in all_failed.values())}")

# Cleanup tmp dir
import shutil
shutil.rmtree(TMP_DIR)
os.makedirs(TMP_DIR, exist_ok=True)
print(f"\n  ✓ Tmp dir cleared")

# Final disk check
result = subprocess.run('df -h .', shell=True, capture_output=True, text=True)
print(f"\n=== DISK SPACE ===")
print(result.stdout.strip())

IMG_SIZE  : 224x224
Splits    : train=24,654  val=3,725  test=6,802
NPZ dir   : ./data/chest/npz
Storage   : float16 images (space efficient)

Building train.npz  (24,654 images)
  [   500/24,654]  elapsed: 0.4m  rate: 24 img/s  ETA: 17.0m
  [ 1,000/24,654]  elapsed: 0.7m  rate: 24 img/s  ETA: 16.2m
  [ 1,500/24,654]  elapsed: 1.0m  rate: 25 img/s  ETA: 15.7m
  [ 2,000/24,654]  elapsed: 1.3m  rate: 25 img/s  ETA: 15.2m
  [ 2,500/24,654]  elapsed: 1.7m  rate: 25 img/s  ETA: 14.8m
  [ 3,000/24,654]  elapsed: 2.0m  rate: 25 img/s  ETA: 14.4m
  [ 3,500/24,654]  elapsed: 2.3m  rate: 25 img/s  ETA: 14.1m
  [ 4,000/24,654]  elapsed: 2.7m  rate: 25 img/s  ETA: 13.7m
  [ 4,500/24,654]  elapsed: 3.0m  rate: 25 img/s  ETA: 13.4m
  [ 5,000/24,654]  elapsed: 3.3m  rate: 25 img/s  ETA: 13.1m
  [ 5,500/24,654]  elapsed: 3.7m  rate: 25 img/s  ETA: 12.7m
  [ 6,000/24,654]  elapsed: 4.0m  rate: 25 img/s  ETA: 12.4m
  [ 6,500/24,654]  elapsed: 4.3m  rate: 25 img/s  ETA: 12.1m
  [ 7,000/24,654]  elapsed: 

## Utility Split

In [10]:
import numpy as np
import json

NPZ_DIR = './data/chest/npz'

with open('./data/metadata/class_to_idx.json') as f:
    class_to_idx = json.load(f)
idx_to_class = {v: k for k, v in class_to_idx.items()}

print("=" * 60)
print("NPZ INTEGRITY CHECK")
print("=" * 60)

for split in ['train', 'val', 'test']:
    data = np.load(f'{NPZ_DIR}/{split}.npz')
    
    imgs  = data['images']
    labs  = data['labels']
    subs  = data['subject_ids']
    studs = data['study_ids']
    gens  = data['genders']

    print(f"\n── {split.upper()} ──────────────────────────────────────")
    print(f"  images      : {imgs.shape}  dtype={imgs.dtype}")
    print(f"  labels      : {labs.shape}   dtype={labs.dtype}")
    print(f"  subject_ids : {subs.shape}   dtype={subs.dtype}")
    print(f"  study_ids   : {studs.shape}   dtype={studs.dtype}")
    print(f"  genders     : {gens.shape}   dtype={gens.dtype}")

    # Value range check
    imgs_f32 = imgs.astype(np.float32)
    print(f"\n  Pixel range : min={imgs_f32.min():.4f}  "
          f"max={imgs_f32.max():.4f}  "
          f"mean={imgs_f32.mean():.4f}  "
          f"std={imgs_f32.std():.4f}")

    # Label distribution
    print(f"\n  Class distribution:")
    unique, counts = np.unique(labs, return_counts=True)
    for idx, cnt in zip(unique, counts):
        pct = cnt / len(labs) * 100
        print(f"    {idx:2d}  {idx_to_class[idx]:<22} {cnt:>5,}  ({pct:.1f}%)")

    # Gender distribution
    m_count = (gens == 1).sum()
    f_count = (gens == 0).sum()
    print(f"\n  Gender : M={m_count:,} ({m_count/len(gens)*100:.1f}%)  "
          f"F={f_count:,} ({f_count/len(gens)*100:.1f}%)")

    # Patient leakage check
    print(f"\n  Unique patients : {len(np.unique(subs)):,}")

    # Check no all-zero or all-one images (corrupted)
    zero_imgs = (imgs_f32.max(axis=(1,2)) == 0).sum()
    ones_imgs = (imgs_f32.min(axis=(1,2)) == 1).sum()
    print(f"  All-zero images : {zero_imgs}")
    print(f"  All-ones images : {ones_imgs}")

    data.close()

# Cross-split patient leakage check
print(f"\n{'='*60}")
print("CROSS-SPLIT PATIENT LEAKAGE CHECK")
print(f"{'='*60}")
train_data = np.load(f'{NPZ_DIR}/train.npz')
val_data   = np.load(f'{NPZ_DIR}/val.npz')
test_data  = np.load(f'{NPZ_DIR}/test.npz')

train_pats = set(train_data['subject_ids'])
val_pats   = set(val_data['subject_ids'])
test_pats  = set(test_data['subject_ids'])

tv_overlap  = train_pats & val_pats
tt_overlap  = train_pats & test_pats
vt_overlap  = val_pats   & test_pats

print(f"  Train ∩ Val  : {len(tv_overlap)}  {'✓' if len(tv_overlap)==0 else '✗ LEAKAGE'}")
print(f"  Train ∩ Test : {len(tt_overlap)}  {'✓' if len(tt_overlap)==0 else '✗ LEAKAGE'}")
print(f"  Val   ∩ Test : {len(vt_overlap)}  {'✓' if len(vt_overlap)==0 else '✗ LEAKAGE'}")

train_data.close()
val_data.close()
test_data.close()

print(f"\n{'='*60}")
print("DATA PREPARATION COMPLETE")
print(f"{'='*60}")
print(f"  train.npz : ./data/chest/npz/train.npz")
print(f"  val.npz   : ./data/chest/npz/val.npz")
print(f"  test.npz  : ./data/chest/npz/test.npz")
print(f"\n  Ready for:")
print(f"    → Raw baseline training (ResNet18 / DenseNet121)")
print(f"    → GS transformation pipeline (gs0–gs50)")
print(f"    → Gender-stratified evaluation")

NPZ INTEGRITY CHECK

── TRAIN ──────────────────────────────────────
  images      : (24654, 224, 224)  dtype=float16
  labels      : (24654,)   dtype=int64
  subject_ids : (24654,)   dtype=int64
  study_ids   : (24654,)   dtype=int64
  genders     : (24654,)   dtype=int8

  Pixel range : min=0.0000  max=1.0000  mean=0.4951  std=0.2475

  Class distribution:
     0  No Finding             4,276  (17.3%)
     1  Infiltration           6,661  (27.0%)
     2  Atelectasis            2,955  (12.0%)
     3  Effusion               2,774  (11.3%)
     4  Nodule                 1,916  (7.8%)
     5  Pneumothorax           1,481  (6.0%)
     6  Mass                   1,507  (6.1%)
     7  Consolidation            944  (3.8%)
     8  Pleural_Thickening       776  (3.1%)
     9  Cardiomegaly             739  (3.0%)
    10  Emphysema                625  (2.5%)

  Gender : M=13,768 (55.8%)  F=10,886 (44.2%)

  Unique patients : 9,436
  All-zero images : 0
  All-ones images : 0

── VAL ──────────────

## ReID Split

In [1]:
import numpy as np
import pandas as pd
import json
from pathlib import Path

META_DIR = Path('./data/metadata')

# Load the filtered metadata CSV — this has all 35,181 images
final_df = pd.read_csv(META_DIR / 'final_filtered.csv')

print(f"=== FILTERED DATASET ===")
print(f"  Total images   : {len(final_df):,}")
print(f"  Total patients : {final_df['patient_id'].nunique():,}")

# Images per patient distribution
imgs_per_patient = final_df.groupby('patient_id').size()

print(f"\n=== IMAGES PER PATIENT DISTRIBUTION ===")
for n in sorted(imgs_per_patient.unique()):
    count = (imgs_per_patient == n).sum()
    pct   = count / len(imgs_per_patient) * 100
    print(f"  {n:3d} image(s) : {count:>5,} patients  ({pct:.1f}%)")

print(f"\n=== SUMMARY ===")
print(f"  Patients with exactly 1 image : "
      f"{(imgs_per_patient == 1).sum():,}  "
      f"({(imgs_per_patient == 1).mean()*100:.1f}%)")
print(f"  Patients with 2+ images       : "
      f"{(imgs_per_patient >= 2).sum():,}  "
      f"({(imgs_per_patient >= 2).mean()*100:.1f}%)")
print(f"  Patients with 3+ images       : "
      f"{(imgs_per_patient >= 3).sum():,}  "
      f"({(imgs_per_patient >= 3).mean()*100:.1f}%)")
print(f"  Patients with 5+ images       : "
      f"{(imgs_per_patient >= 5).sum():,}  "
      f"({(imgs_per_patient >= 5).mean()*100:.1f}%)")
print(f"  Max images per patient        : {imgs_per_patient.max():,}")
print(f"  Mean images per patient       : {imgs_per_patient.mean():.2f}")
print(f"  Median images per patient     : {imgs_per_patient.median():.1f}")

# How many images come from multi-image patients
multi_img_patients = imgs_per_patient[imgs_per_patient >= 2].index
multi_img_df = final_df[final_df['patient_id'].isin(multi_img_patients)]
single_img_df = final_df[~final_df['patient_id'].isin(multi_img_patients)]

print(f"\n=== IMAGES FROM MULTI-IMAGE PATIENTS (2+) ===")
print(f"  Images : {len(multi_img_df):,}  "
      f"({len(multi_img_df)/len(final_df)*100:.1f}% of total)")
print(f"  Patients : {multi_img_df['patient_id'].nunique():,}")

print(f"\n=== IMAGES FROM SINGLE-IMAGE PATIENTS ===")
print(f"  Images   : {len(single_img_df):,}  "
      f"({len(single_img_df)/len(final_df)*100:.1f}% of total)")
print(f"  Patients : {single_img_df['patient_id'].nunique():,}")

# Class distribution among multi-image patients
print(f"\n=== CLASS DISTRIBUTION — MULTI-IMAGE PATIENTS ===")
with open(META_DIR / 'class_to_idx.json') as f:
    class_to_idx = json.load(f)
idx_to_class = {v: k for k, v in class_to_idx.items()}

class_counts = multi_img_df['finding_labels'].value_counts()
for cls, cnt in class_counts.items():
    pct = cnt / len(multi_img_df) * 100
    print(f"  {cls:<22} {cnt:>5,}  ({pct:.1f}%)")

# Gender distribution among multi-image patients
print(f"\n=== GENDER — MULTI-IMAGE PATIENTS ===")
gender_counts = multi_img_df['patient_gender'].value_counts()
print(gender_counts.to_string())
print((gender_counts / len(multi_img_df)).round(3).to_string())

=== FILTERED DATASET ===
  Total images   : 35,181
  Total patients : 13,481

=== IMAGES PER PATIENT DISTRIBUTION ===
    1 image(s) : 7,979 patients  (59.2%)
    2 image(s) : 2,057 patients  (15.3%)
    3 image(s) : 1,084 patients  (8.0%)
    4 image(s) :   622 patients  (4.6%)
    5 image(s) :   396 patients  (2.9%)
    6 image(s) :   280 patients  (2.1%)
    7 image(s) :   214 patients  (1.6%)
    8 image(s) :   144 patients  (1.1%)
    9 image(s) :   102 patients  (0.8%)
   10 image(s) :   107 patients  (0.8%)
   11 image(s) :    78 patients  (0.6%)
   12 image(s) :    51 patients  (0.4%)
   13 image(s) :    47 patients  (0.3%)
   14 image(s) :    42 patients  (0.3%)
   15 image(s) :    36 patients  (0.3%)
   16 image(s) :    33 patients  (0.2%)
   17 image(s) :    27 patients  (0.2%)
   18 image(s) :    21 patients  (0.2%)
   19 image(s) :    23 patients  (0.2%)
   20 image(s) :    11 patients  (0.1%)
   21 image(s) :    17 patients  (0.1%)
   22 image(s) :    14 patients  (0.1%)


In [4]:
import numpy as np
import pandas as pd
import json
from pathlib import Path

META_DIR      = Path('./data/metadata')
REID_META_DIR = Path('./data/metadata/reid')
REID_META_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(42)

# Load filtered metadata
final_df = pd.read_csv(META_DIR / 'final_filtered.csv')

# ── Step 1: Keep only patients with 5+ images ─────────────────────────────────
imgs_per_patient = final_df.groupby('patient_id').size()
eligible_patients = imgs_per_patient[
    imgs_per_patient >= 5
].index.tolist()

eligible_df = final_df[
    final_df['patient_id'].isin(eligible_patients)
].copy()

print(f"=== ELIGIBLE PATIENTS (5+ images) ===")
print(f"  Patients : {len(eligible_patients):,}")
print(f"  Images   : {len(eligible_df):,}")

# ── Step 2: Assign roles per patient ──────────────────────────────────────────
train_rows = []
val_rows   = []
test_rows  = []

for pid in eligible_patients:
    patient_imgs = eligible_df[
        eligible_df['patient_id'] == pid
    ].sort_values('followup_num').reset_index(drop=True)

    n = len(patient_imgs)

    # First 4 → 3 train + 1 val (random position within first 4)
    first_four = patient_imgs.iloc[:4]
    val_idx    = np.random.randint(0, 4)
    val_img    = first_four.iloc[[val_idx]]
    train_imgs = first_four.drop(first_four.index[val_idx])

    # Images 5–6 → test (cap at 2)
    n_test    = min(n - 4, 2)
    test_imgs = patient_imgs.iloc[4:4 + n_test]

    for _, row in train_imgs.iterrows():
        train_rows.append({**row.to_dict(), 'reid_split': 'train'})
    for _, row in val_img.iterrows():
        val_rows.append({**row.to_dict(), 'reid_split': 'val'})
    for _, row in test_imgs.iterrows():
        test_rows.append({**row.to_dict(), 'reid_split': 'test'})

train_df = pd.DataFrame(train_rows)
val_df   = pd.DataFrame(val_rows)
test_df  = pd.DataFrame(test_rows)

# ── Step 3: Sanity checks ──────────────────────────────────────────────────────
n_eligible = len(eligible_patients)

assert train_df['patient_id'].nunique() == n_eligible, \
    "Not all patients in train"
assert val_df['patient_id'].nunique() == n_eligible, \
    "Not all patients in val"
assert test_df['patient_id'].nunique() == n_eligible, \
    "Not all patients in test"
print(f"✓ All {n_eligible:,} patients appear in all three splits")

# Exactly 3 train images per patient
train_per_patient = train_df.groupby('patient_id').size()
assert (train_per_patient == 3).all(), \
    f"Train should have exactly 3 per patient, got: {train_per_patient.value_counts()}"
print(f"✓ Train has exactly 3 images per patient")

# Exactly 1 val image per patient
val_per_patient = val_df.groupby('patient_id').size()
assert (val_per_patient == 1).all(), \
    "Val should have exactly 1 image per patient"
print(f"✓ Val has exactly 1 image per patient")

# No image overlap across splits
train_imgs_set = set(train_df['image_index'])
val_imgs_set   = set(val_df['image_index'])
test_imgs_set  = set(test_df['image_index'])
assert len(train_imgs_set & val_imgs_set)  == 0, "OVERLAP: train/val"
assert len(train_imgs_set & test_imgs_set) == 0, "OVERLAP: train/test"
assert len(val_imgs_set   & test_imgs_set) == 0, "OVERLAP: val/test"
print(f"✓ No image overlap across splits")

# ── Step 4: Summary ────────────────────────────────────────────────────────────
print(f"\n=== REID SPLIT SUMMARY ===")
print(f"  Train : {len(train_df):,} images  "
      f"{train_df['patient_id'].nunique():,} patients  "
      f"({len(train_df)/train_df['patient_id'].nunique():.1f} img/patient)")
print(f"  Val   : {len(val_df):,} images  "
      f"{val_df['patient_id'].nunique():,} patients  "
      f"(1.0 img/patient fixed)")
print(f"  Test  : {len(test_df):,} images  "
      f"{test_df['patient_id'].nunique():,} patients")

print(f"\n=== TEST IMAGES PER PATIENT ===")
test_per_patient = test_df.groupby('patient_id').size()
print(f"  1 test image : {(test_per_patient == 1).sum():,} patients  "
      f"(exactly 5 total images)")
print(f"  2 test images: {(test_per_patient == 2).sum():,} patients  "
      f"(6+ total images)")

print(f"\n=== CLASS DISTRIBUTION ===")
for split_name, split in [('Train', train_df),
                           ('Val',   val_df),
                           ('Test',  test_df)]:
    print(f"\n  {split_name}:")
    for cls, cnt in split['finding_labels'].value_counts().items():
        print(f"    {cls:<22} {cnt:>5,}  ({cnt/len(split)*100:.1f}%)")

print(f"\n=== GENDER DISTRIBUTION ===")
for split_name, split in [('Train', train_df),
                           ('Val',   val_df),
                           ('Test',  test_df)]:
    g = split['patient_gender'].value_counts(normalize=True).round(3)
    print(f"  {split_name}: M={g.get('M',0):.3f}  F={g.get('F',0):.3f}")

# ── Step 5: Save ───────────────────────────────────────────────────────────────
train_df.to_csv(REID_META_DIR / 'reid_train.csv', index=False)
val_df.to_csv(REID_META_DIR   / 'reid_val.csv',   index=False)
test_df.to_csv(REID_META_DIR  / 'reid_test.csv',  index=False)

patient_ids  = sorted(train_df['patient_id'].unique().tolist())
pid_to_label = {pid: idx for idx, pid in enumerate(patient_ids)}

with open(REID_META_DIR / 'pid_to_label.json', 'w') as f:
    json.dump({str(k): v for k, v in pid_to_label.items()}, f, indent=2)

print(f"\n=== SAVED FILES ===")
for f in sorted(REID_META_DIR.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<30} {size_kb:.1f} KB")

print(f"\n  NUM_CLASSES : {len(pid_to_label):,} unique patients")
print(f"\n✓ ReID split complete")

=== ELIGIBLE PATIENTS (5+ images) ===
  Patients : 1,739
  Images   : 17,348
✓ All 1,739 patients appear in all three splits
✓ Train has exactly 3 images per patient
✓ Val has exactly 1 image per patient
✓ No image overlap across splits

=== REID SPLIT SUMMARY ===
  Train : 5,217 images  1,739 patients  (3.0 img/patient)
  Val   : 1,739 images  1,739 patients  (1.0 img/patient fixed)
  Test  : 3,082 images  1,739 patients

=== TEST IMAGES PER PATIENT ===
  1 test image : 396 patients  (exactly 5 total images)
  2 test images: 1,343 patients  (6+ total images)

=== CLASS DISTRIBUTION ===

  Train:
    Infiltration           1,150  (22.0%)
    No Finding             1,007  (19.3%)
    Effusion                 704  (13.5%)
    Atelectasis              664  (12.7%)
    Pneumothorax             431  (8.3%)
    Nodule                   341  (6.5%)
    Mass                     327  (6.3%)
    Consolidation            220  (4.2%)
    Pleural_Thickening       126  (2.4%)
    Emphysema          

### Build ReID NPZs from GCS

In [6]:
import numpy as np
import pandas as pd
import json
import subprocess
import os
import gc
import time
from pathlib import Path
from PIL import Image
from gs_functions import *

# ── CONFIG ────────────────────────────────────────────────────────────────────
GCS_IMG_DIR   = 'gs://nih-cxr-say/nih_cxr/images'
REID_META_DIR = Path('./data/metadata/reid')
REID_NPZ_DIR  = Path('./data/chest/reid_npz')
TMP_DIR       = Path('./data/chest/reid_tmp')
IMG_SIZE      = 224
GS_BATCH_SIZE = 100
GS_ITERATIONS = 50

REID_NPZ_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)

# Conditions — reverse order same as utility
CONDITIONS = ['raw', 'gs50', 'gs40', 'gs30', 'gs20', 'gs10', 'gs0']

# Load split CSVs and pid→label mapping
train_df = pd.read_csv(REID_META_DIR / 'reid_train.csv')
val_df   = pd.read_csv(REID_META_DIR / 'reid_val.csv')
test_df  = pd.read_csv(REID_META_DIR / 'reid_test.csv')

with open(REID_META_DIR / 'pid_to_label.json') as f:
    pid_to_label = {int(k): v for k, v in json.load(f).items()}

print(f"=== REID NPZ BUILDER ===")
print(f"  Train : {len(train_df):,} images")
print(f"  Val   : {len(val_df):,} images")
print(f"  Test  : {len(test_df):,} images")
print(f"  Total : {len(train_df)+len(val_df)+len(test_df):,} images")
print(f"  Conditions : {CONDITIONS}")
print(f"  Output : {REID_NPZ_DIR}")


# ── Helper: download images from GCS and load as numpy ───────────────────────
def fetch_images_from_gcs(
    image_filenames : list,
    patient_ids     : list,
    genders         : list,
    split_name      : str,
) -> tuple:
    """
    Download images from GCS, resize to 224x224 grayscale,
    normalize to [0,1], return as numpy arrays.

    Returns:
        images      : (N, 224, 224) float32
        patient_ids : (N,) int64
        labels      : (N,) int64   pid→integer label
        genders     : (N,) int8
    """
    n       = len(image_filenames)
    images  = np.zeros((n, IMG_SIZE, IMG_SIZE), dtype=np.float32)
    failed  = []

    print(f"  Downloading {n:,} images for {split_name}...")
    t0 = time.time()

    # Download in batches of 500
    BATCH = 500
    for batch_start in range(0, n, BATCH):
        batch_end   = min(batch_start + BATCH, n)
        batch_files = image_filenames[batch_start:batch_end]

        # Build gsutil cp command
        src_paths = ' '.join(
            [f'{GCS_IMG_DIR}/{f}' for f in batch_files]
        )
        result = subprocess.run(
            f'gsutil -m cp {src_paths} {TMP_DIR}/',
            shell=True, capture_output=True, text=True
        )
        if result.returncode != 0:
            print(f"    ⚠ GCS batch error: {result.stderr[:200]}")

        # Load and process each image
        for local_i, fname in enumerate(batch_files):
            global_i = batch_start + local_i
            fpath    = TMP_DIR / fname
            try:
                img = Image.open(fpath).convert('L')
                img = img.resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS)
                images[global_i] = np.array(img, dtype=np.float32) / 255.0
            except Exception as e:
                failed.append((fname, str(e)))
                print(f"    ✗ Failed: {fname} — {e}")
            finally:
                if fpath.exists():
                    os.remove(fpath)

        elapsed = time.time() - t0
        rate    = batch_end / elapsed
        eta     = (n - batch_end) / rate if rate > 0 else 0
        print(f"    [{batch_end:>5,}/{n:,}]  "
              f"{elapsed/60:.1f}m elapsed  "
              f"ETA {eta/60:.1f}m")

    if failed:
        print(f"  ⚠ {len(failed)} images failed to load")

    labels_arr  = np.array(
        [pid_to_label[pid] for pid in patient_ids],
        dtype=np.int64
    )
    pids_arr    = np.array(patient_ids, dtype=np.int64)
    genders_arr = np.array(genders,     dtype=np.int8)

    # Verify no wrong patient mapping
    assert len(images) == len(labels_arr) == len(pids_arr), \
        "Length mismatch between images and labels"

    return images, pids_arr, labels_arr, genders_arr


# ── Helper: apply GS transformation ──────────────────────────────────────────
def apply_gs(images: np.ndarray, condition: str) -> np.ndarray:
    if condition == 'raw':
        return images.copy()
    mask_pct = int(condition.replace('gs', '')) / 100.0
    print(f"    Applying GS maskP={mask_pct}  "
          f"ite={GS_ITERATIONS}  batch={GS_BATCH_SIZE}...")
    t0  = time.time()
    out = GS_batch_image(
        images,
        batch_size = GS_BATCH_SIZE,
        ite        = GS_ITERATIONS,
        maskP      = mask_pct,
    ).astype(np.float32)
    print(f"    GS done in {time.time()-t0:.1f}s  "
          f"range=[{out.min():.3f}, {out.max():.3f}]")
    return out


# ── Helper: check if NPZ already exists ──────────────────────────────────────
def npz_path(split: str, condition: str) -> Path:
    return REID_NPZ_DIR / f'reid_{split}_{condition}.npz'

def npz_exists(split: str, condition: str) -> bool:
    p = npz_path(split, condition)
    if not p.exists():
        return False
    size_mb = p.stat().st_size / 1e6
    if size_mb < 1.0:
        print(f"  ⚠ Suspicious small file: {p.name} ({size_mb:.1f} MB) — rebuilding")
        return False
    return True


# =============================================================================
# MAIN BUILD LOOP
# =============================================================================

SPLIT_DFS = {
    'train': train_df,
    'val'  : val_df,
    'test' : test_df,
}

# ── Step 1: Download raw images once per split ────────────────────────────────
print("\n" + "="*70)
print("STEP 1 — Download raw images from GCS")
print("="*70)

raw_images = {}   # split → (images, pids, labels, genders)

for split_name, split_df in SPLIT_DFS.items():
    # Sort by image_index for deterministic ordering
    split_df = split_df.sort_values('image_index').reset_index(drop=True)
    SPLIT_DFS[split_name] = split_df   # update with sorted version

    # Check if all conditions already exist for this split
    all_exist = all(
        npz_exists(split_name, cond) for cond in CONDITIONS
    )
    if all_exist:
        print(f"\n  ✓ All conditions exist for {split_name} — skip download")
        # Still need raw images for GS transform if any condition missing
        continue

    print(f"\n  Fetching {split_name} ({len(split_df):,} images)...")

    gender_map = {'M': 1, 'F': 0}
    images, pids, labels, genders = fetch_images_from_gcs(
        image_filenames = split_df['image_index'].tolist(),
        patient_ids     = split_df['patient_id'].tolist(),
        genders         = [gender_map[g]
                           for g in split_df['patient_gender'].tolist()],
        split_name      = split_name,
    )

    # Verify patient→label mapping is correct
    for i in range(min(5, len(pids))):
        expected = pid_to_label[int(pids[i])]
        assert labels[i] == expected, \
            f"Label mismatch at index {i}: " \
            f"pid={pids[i]} expected={expected} got={labels[i]}"
    print(f"  ✓ Patient→label mapping verified (spot check passed)")

    raw_images[split_name] = (images, pids, labels, genders)

    print(f"  images : {images.shape}  "
          f"range=[{images.min():.3f}, {images.max():.3f}]")


# ── Step 2: Build NPZs per condition ─────────────────────────────────────────
print("\n" + "="*70)
print("STEP 2 — Build NPZs per condition")
print("="*70)

total_t0 = time.time()

for condition in CONDITIONS:
    print(f"\n── CONDITION: {condition} ──────────────────────────────────────")

    for split_name in ['train', 'val', 'test']:
        out_path = npz_path(split_name, condition)

        if npz_exists(split_name, condition):
            size_mb = out_path.stat().st_size / 1e6
            print(f"  ✓ EXISTS  {out_path.name:<40} ({size_mb:.0f} MB) — skip")
            continue

        if split_name not in raw_images:
            print(f"  ✗ Raw images not loaded for {split_name} — cannot build")
            continue

        images, pids, labels, genders = raw_images[split_name]

        # Apply GS transformation
        transformed = apply_gs(images, condition)

        # Save NPZ
        np.savez_compressed(
            out_path,
            images      = transformed,
            labels      = labels,
            patient_ids = pids,
            genders     = genders,
        )

        size_mb = out_path.stat().st_size / 1e6
        print(f"  ✓ Saved  {out_path.name:<40} ({size_mb:.0f} MB)")

        del transformed
        gc.collect()

total_elapsed = (time.time() - total_t0) / 60
print(f"\n{'='*70}")
print(f"BUILD COMPLETE — {total_elapsed:.1f} min total")
print(f"{'='*70}")

# ── Step 3: Disk summary ──────────────────────────────────────────────────────
print(f"\n=== NPZ FILES ON DISK ===")
total_size = 0
for f in sorted(REID_NPZ_DIR.iterdir()):
    size_mb = f.stat().st_size / 1e6
    total_size += size_mb
    print(f"  {f.name:<45} {size_mb:>6.0f} MB")
print(f"\n  Total : {total_size/1e3:.2f} GB")

# ── Step 4: Integrity check on raw NPZs ──────────────────────────────────────
print(f"\n=== INTEGRITY CHECK (raw NPZs) ===")
for split_name in ['train', 'val', 'test']:
    p = npz_path(split_name, 'raw')
    if not p.exists():
        print(f"  ✗ Missing: {p.name}")
        continue
    data = np.load(p)
    imgs  = data['images']
    labs  = data['labels']
    pids  = data['patient_ids']
    gens  = data['genders']

    # Check unique patients match expected
    n_unique = len(np.unique(pids))
    print(f"\n  {split_name}:")
    print(f"    images      : {imgs.shape}  dtype={imgs.dtype}")
    print(f"    labels      : {labs.shape}  "
          f"range=[{labs.min()},{labs.max()}]")
    print(f"    patient_ids : {pids.shape}  "
          f"unique={n_unique}")
    print(f"    genders     : {gens.shape}  "
          f"M={(gens==1).sum():,} F={(gens==0).sum():,}")
    print(f"    pixel range : [{imgs.min():.3f}, {imgs.max():.3f}]")

    # Verify label consistency
    for i in range(min(10, len(pids))):
        expected = pid_to_label[int(pids[i])]
        assert labs[i] == expected, \
            f"Label mismatch: pid={pids[i]} " \
            f"expected={expected} got={labs[i]}"
    print(f"    label check : ✓ (spot check passed)")
    data.close()

# Clean up tmp dir
import shutil
shutil.rmtree(TMP_DIR)
TMP_DIR.mkdir(exist_ok=True)
print(f"\n✓ Tmp dir cleared")
print(f"✓ ReID NPZs ready at {REID_NPZ_DIR}")

=== REID NPZ BUILDER ===
  Train : 5,217 images
  Val   : 1,739 images
  Test  : 3,082 images
  Total : 10,038 images
  Conditions : ['raw', 'gs50', 'gs40', 'gs30', 'gs20', 'gs10', 'gs0']
  Output : data/chest/reid_npz

STEP 2 — Build NPZs per condition

── CONDITION: raw ──────────────────────────────────────
  ✓ EXISTS  reid_train_raw.npz                       (235 MB) — skip
  ✓ EXISTS  reid_val_raw.npz                         (78 MB) — skip
  ✓ EXISTS  reid_test_raw.npz                        (138 MB) — skip

── CONDITION: gs50 ──────────────────────────────────────
    Applying GS maskP=0.5  ite=50  batch=100...
Batch 52 of 52 (full batch) completed...
Remaining data batch completed (size: 17)...    GS done in 311.5s  range=[0.000, 2.988]
  ✓ Saved  reid_train_gs50.npz                      (930 MB)
    Applying GS maskP=0.5  ite=50  batch=100...
Batch 17 of 17 (full batch) completed...
Remaining data batch completed (size: 39)...    GS done in 96.4s  range=[0.000, 2.874]
  ✓ Saved